# FAISS

Faiss is a library for efficient similarity search and clustering of dense vectors. It contains algorithms that search in sets of vectors of any size, up to ones that possibly do not fit in RAM. It also contains supporting code for evaluation and parameter tuning.

In [7]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

In [12]:
# Data ingestion & transformartion

loader = TextLoader("speech.txt")
documents = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=30 )
docs =text_splitter.split_documents(documents)

In [11]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\nâ€¦'),
 Document(metadata={'source': 'speech.txt'}, page_content='â€¦\n\nIt will be all the easier for us to conduct

In [17]:
# embeddingd

embedding = OllamaEmbeddings(model="nomic-embed-text")
db = FAISS.from_documents(docs,embedding)
db

In [20]:
# Querying

query = "what does the speaker desired outcome of the war?"
docs = db.similarity_search(query)
docs[0].page_content


'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### As a Retriever
We can also convert the vectorstore into a Retriever class. This allows us to easily use it in other LangChain methods, which largely work with retrievers

In [21]:
retriever=db.as_retriever()
docs=retriever.invoke(query)
docs[0].page_content

'It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'

#### Similarity Search with score
There are some FAISS specific methods. One of them is similarity_search_with_score, which allows you to return not only the documents but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better.

In [22]:
docs_and_score=db.similarity_search_with_score(query)
docs_and_score

[(Document(id='baae0797-71b0-4da2-8cc2-960d040b41ab', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
  np.float32(367.003)),
 (Document(id='777069aa-2ae7-4873-92fa-5b491ef0082c', metadata={'source': 'spe

In [24]:
embedding_vector=embedding.embed_query(query)
embedding_vector

[-0.4397379755973816,
 0.25245723128318787,
 -3.5799400806427,
 -0.7401418089866638,
 1.776535153388977,
 1.8206130266189575,
 -1.420017123222351,
 0.2780938148498535,
 0.9427220225334167,
 -0.333130419254303,
 -0.04572800174355507,
 0.8492722511291504,
 0.637544572353363,
 1.6974003314971924,
 2.10681414604187,
 -1.0560814142227173,
 0.33219969272613525,
 -0.9592995047569275,
 -0.8368968963623047,
 0.45811665058135986,
 -1.0122966766357422,
 -0.6027417182922363,
 -0.18360717594623566,
 0.29966920614242554,
 1.5490920543670654,
 0.2232707291841507,
 -0.09829231351613998,
 0.6981585025787354,
 -1.0054491758346558,
 -0.10211556404829025,
 0.9844894409179688,
 0.07906261831521988,
 -0.10442817211151123,
 0.07764992862939835,
 -1.2854739427566528,
 -1.8423594236373901,
 0.2169853150844574,
 0.9367972016334534,
 0.5126363039016724,
 -1.2977994680404663,
 0.2898116111755371,
 -0.4215433597564697,
 -0.18656957149505615,
 -1.7967969179153442,
 1.4973187446594238,
 -0.5588234663009644,
 -0.1325

In [25]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='baae0797-71b0-4da2-8cc2-960d040b41ab', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='777069aa-2ae7-4873-92fa-5b491ef0082c', metadata={'source': 'speech.txt'}, page_content='â

In [26]:
### Saving And Loading
db.save_local("faiss_index")

In [28]:
new_db=FAISS.load_local("faiss_index",embedding,allow_dangerous_deserialization=True)
docs=new_db.similarity_search(query)

In [29]:
docs

[Document(id='baae0797-71b0-4da2-8cc2-960d040b41ab', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. There are, it may be, many months of fiery trial and sacrifice ahead of us. It is a fearful thing to lead this great peaceful people into war, into the most terrible and disastrous of all wars, civilization itself seeming to be in the balance. But the right is more precious than peace, and we shall fight for the things which we have always carried nearest our heartsâ€”for democracy, for the right of those who submit to authority to have a voice in their own governments, for the rights and liberties of small nations, for a universal dominion of right by such a concert of free peoples as shall bring peace and safety to all nations and make the world itself at last free.'),
 Document(id='777069aa-2ae7-4873-92fa-5b491ef0082c', metadata={'source': 'speech.txt'}, page_content='â